# Part 2 — Working with NCBI in Python
## Beginner Edition · Day 2

**Course:** Introduction to Bioinformatics — HCT Cohort
**Instructor:** Norah AlGhamdi

---

### What you'll do
In Part 1, you went to NCBI's website by hand. Now Python will do the same for you — and a lot more.

By the end you will have:

- ⬇️ Downloaded a real gene from NCBI in code
- 💾 Saved it to a file
- 🧬 Pulled the matching protein
- 📊 Calculated basic stats
- 🌍 Compared a gene across species
- 🔎 Searched NCBI by keyword
- 🔁 Downloaded multiple genes at once

### How to use this notebook
1. **Save your own copy first:** `File → Save a copy in Drive`
2. **Run each cell with `Shift + Enter`**
3. Most cells are written for you — just run them.
4. **✏️ Your Turn** cells need a small change from you.

Let's go. 🚀


---
## Setup — Install Biopython

We only need to run this once at the start of the session.


In [ ]:
!pip install biopython --quiet


Now tell Python which parts of Biopython we'll use:


In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

# NCBI asks who you are (any email works for learning)
Entrez.email = "your_email@example.com"

print("Ready!")


---
## Task 1 — Download a real gene from NCBI

Remember in Part 1 when you went to NCBI's website to get the BRCA1 FASTA?
**Python can do that for you in 4 lines.**

Let's download the BRCA1 mRNA (accession **NM_007294**):


In [ ]:
handle = Entrez.efetch(db="nucleotide", id="NM_007294", rettype="fasta", retmode="text")
fasta_data = handle.read()
handle.close()

# Show the first 300 characters
print(fasta_data[:300])


**What just happened?** Python connected to NCBI, requested the BRCA1 sequence, and got it back. Same data you downloaded by hand in Part 1 — but in seconds, from inside your code.


---
## Task 2 — Save it to a file

Let's keep what we just downloaded:


In [ ]:
with open("BRCA1.fasta", "w") as f:
    f.write(fasta_data)

print("Saved BRCA1.fasta")


Now read it back properly with Biopython so we can use it:


In [ ]:
for record in SeqIO.parse("BRCA1.fasta", "fasta"):
    print("ID:         ", record.id)
    print("Description:", record.description[:80], "...")
    print("Length:     ", len(record.seq), "bases")


---
## Task 3 — ✏️ Your Turn — Your own gene

In Part 1, you noted your own gene's accession number (something like `NM_xxxxxx`).

**Replace `NM_007294` below with your gene's accession, then run the cell.**


In [ ]:
# Change this to your gene's accession from Part 1
my_accession = "NM_007294"   # ← replace this

handle = Entrez.efetch(db="nucleotide", id=my_accession, rettype="fasta", retmode="text")
my_fasta = handle.read()
handle.close()

# Save it
with open("my_gene.fasta", "w") as f:
    f.write(my_fasta)

# Print the first 300 characters
print(my_fasta[:300])


---
## Task 4 — Get the protein for your gene

Every gene that codes for a protein has a matching protein record — its accession starts with **NP_**.

For BRCA1, the protein is **NP_009225**. We download it the same way — but from the `protein` database instead of `nucleotide`:


In [ ]:
handle = Entrez.efetch(db="protein", id="NP_009225", rettype="fasta", retmode="text")
protein_data = handle.read()
handle.close()

# Save and read
with open("BRCA1_protein.fasta", "w") as f:
    f.write(protein_data)

for record in SeqIO.parse("BRCA1_protein.fasta", "fasta"):
    print("Protein ID:    ", record.id)
    print("Description:   ", record.description[:80])
    print("Length:        ", len(record.seq), "amino acids")
    print("First 30 aas:  ", record.seq[:30])


Notice: the protein has only **single letters of amino acids** (M, D, L, S, A, …) — not A, T, G, C.

And the protein is **much shorter than the DNA**, because every 3 DNA bases (a codon) become 1 amino acid.


### ✏️ Your Turn
From Part 1, write your own gene's protein accession (the **NP_** number). Replace it below and run.


In [ ]:
my_protein = "NP_009225"   # ← replace with your protein accession

handle = Entrez.efetch(db="protein", id=my_protein, rettype="fasta", retmode="text")
my_protein_data = handle.read()
handle.close()

print(my_protein_data[:300])


---
## Task 5 — Calculate basic stats

Now that we have the BRCA1 sequence in a file, let's compute a quick "data card" for it.


In [ ]:
for record in SeqIO.parse("BRCA1.fasta", "fasta"):
    seq = record.seq
    length = len(seq)
    a = seq.count("A")
    t = seq.count("T")
    g = seq.count("G")
    c = seq.count("C")
    gc = round(gc_fraction(seq) * 100, 2)
    
    print("Gene:        ", record.id)
    print("Length:      ", length, "bases")
    print("A:           ", a)
    print("T:           ", t)
    print("G:           ", g)
    print("C:           ", c)
    print("GC content:  ", gc, "%")


That's the same kind of analysis we did in Day 1 — but now on a real, freshly-downloaded gene from NCBI.


---
## Task 6 — Compare a gene across two species

BRCA1 isn't only in humans. Mice have it too — it has a slightly different version. Let's compare them.

- **Human BRCA1:** `NM_007294`
- **Mouse Brca1:** `NM_009764`

We'll download both and compare their lengths and GC content.


In [ ]:
# Download both at once
handle = Entrez.efetch(db="nucleotide", id="NM_007294,NM_009764", rettype="fasta", retmode="text")
both_fastas = handle.read()
handle.close()

# Save to one file with both records inside
with open("brca1_two_species.fasta", "w") as f:
    f.write(both_fastas)

print("Saved both!")


Now let's read them and compare:


In [ ]:
print("Comparing BRCA1 across species:")
print("---")
for record in SeqIO.parse("brca1_two_species.fasta", "fasta"):
    length = len(record.seq)
    gc = round(gc_fraction(record.seq) * 100, 2)
    print(record.id)
    print("  Length:    ", length, "bases")
    print("  GC content:", gc, "%")
    print()


**Observe:** the lengths and GC contents are similar but not identical. Real evolution at work — same gene, slightly different versions in different species.


---
## Task 7 — Search NCBI by keyword

What if you don't know the accession number — but you know the **gene name**?

Python can search NCBI for you. Let's search for **insulin** in human:


In [ ]:
# Search NCBI for the human insulin mRNA
search_handle = Entrez.esearch(db="nucleotide", term="INS[Gene Name] AND human[Organism] AND mRNA[Filter]", retmax=3)
search_results = Entrez.read(search_handle)
search_handle.close()

# Show the first 3 matching IDs
print("Found these matching records:")
print(search_results["IdList"])


Those are NCBI's internal IDs (called UIDs) for the top 3 matching insulin records.

Let's grab the first one and download its sequence:


In [ ]:
first_id = search_results["IdList"][0]

handle = Entrez.efetch(db="nucleotide", id=first_id, rettype="fasta", retmode="text")
insulin_fasta = handle.read()
handle.close()

print(insulin_fasta[:300])


**Powerful idea:** you went from a search keyword ("INS gene, human, mRNA") to the actual sequence — without ever opening a web browser.


---
## Task 8 — Download multiple genes in one loop

This is where Python really shines. Manual NCBI downloads = one at a time.
Python = 5 genes in 5 seconds.

Let's grab 5 famous human genes:


In [ ]:
# A list of 5 famous gene accessions
gene_list = {
    "BRCA1": "NM_007294",
    "TP53":  "NM_000546",
    "INS":   "NM_000207",
    "HBB":   "NM_000518",
    "GFP":   "NM_001293557"
}

print("We will download:", list(gene_list.keys()))


Now loop and download each one:


In [ ]:
all_records = ""

for name, accession in gene_list.items():
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
    fasta = handle.read()
    handle.close()
    all_records += fasta
    print("Downloaded", name)

# Save them all to one file
with open("five_genes.fasta", "w") as f:
    f.write(all_records)

print()
print("All 5 saved to five_genes.fasta!")


Now let's see all 5 at once with their stats:


In [ ]:
print("Gene comparison:")
print("---")
for record in SeqIO.parse("five_genes.fasta", "fasta"):
    length = len(record.seq)
    gc = round(gc_fraction(record.seq) * 100, 1)
    print(f"{record.id:18}  length={length:>6} bases   GC={gc}%")


**Look at that table.** You just compared 5 real human genes in one shot. Doing this manually on the website would have taken 15+ minutes — Python did it in seconds.

*This is the power of code.*


---
## 📚 Homework — Apply What You Learned Yesterday

Today we downloaded real genes from NCBI. Yesterday we learned how to **count bases, find motifs, and compare sequences**.

Let's combine both! Below are 3 small homework tasks that re-use the tricks from Day 1 on the data you just pulled in Task 6 and Task 8.

**Hint:** look back at your Day 1 notebooks if you need to remember any function.


---
### 🏠 Homework 1 — How different are human and mouse BRCA1?

In Task 6, you downloaded both human (`NM_007294`) and mouse (`NM_009764`) BRCA1 into the file `brca1_two_species.fasta`.

Yesterday you learned that when two sequences are the **same length**, you can count how many positions differ — that's the **Hamming distance**.

**Your task:** look at the **first 100 bases** of each species and count how many positions differ.

Most of the code is written for you — just fill in the blank.


In [ ]:
# Read both species into a list
records = list(SeqIO.parse("brca1_two_species.fasta", "fasta"))

human_seq = str(records[0].seq)[:100]   # first 100 bases of human
mouse_seq = str(records[1].seq)[:100]   # first 100 bases of mouse

print("Human:", human_seq)
print("Mouse:", mouse_seq)


Now count how many positions differ — using the same pattern from yesterday:


In [ ]:
# Count differences between human and mouse



**What does this tell us?** 


---
### 🏠 Homework 2 (bonus) — Where is the FIRST start codon?

Yesterday you used `.find()` to locate where a motif first appears in a sequence.

**Find the position of the first `ATG` in human BRCA1.**


In [ ]:
# Find the position of the first ATG


**Cool fact:** that first ATG is where the BRCA1 protein actually starts being made. The bases that come right after it code for the first 10 amino acids of the BRCA1 protein.


---
### 🏠 Optional Challenge — Compare GC content across all 5 genes

In Task 8, you downloaded 5 famous genes into `five_genes.fasta`.

**Which gene has the highest GC content? Which has the lowest?**

(All the code you need is from yesterday + today — try without looking, then check the cell.)


---
### Submission

Bring these answers to next Friday's session:

1. How many differences in the first 100 bases of BRCA1 between human and mouse?
2. How many of each stop codon (TAA, TAG, TGA) in human BRCA1?
3. Where is the first ATG in human BRCA1?
4. (Bonus) Which gene has the highest GC%?

*Even if you can't finish all four — just try the first one. Effort matters more than perfection.*


---
## 🎉 You did it!

You just learned to do — in Python — what most biologists do by hand:

| What you did | Real-world skill |
|---|---|
| Task 1: Download a gene | Pulling sequences for analysis |
| Task 2: Save & re-read | Building reproducible workflows |
| Task 3: Your own gene | Personal exploration |
| Task 4: Pull the protein | Linking gene → protein |
| Task 5: Calculate stats | Quality-checking your data |
| Task 6: Compare species | Comparative genomics |
| Task 7: Search by keyword | Finding data without an accession |
| Task 8: Many genes at once | Scaling up to real research |

### Coming up next weekend

Now that you can pull and inspect sequences, next weekend we'll **align** them, **search** for similar sequences with BLAST, and start finding biological patterns.

**Great work today. See you next Friday!** 👋
